# 02 — Shared Classifier Pilot and Official Baselines

Follow the adopted master plan: three learning rates × two settings × B1/B3 =
**12 pilot fits**. Select one rate by the equal-weight mean of the four
setting/condition scores, then freeze the shared configuration before main runs.

Device selection is **CUDA → MPS → CPU**. All runs use float32 and the same
training policy. Large weights and detailed outputs stay under `BASE_PATH`.
The full pilot is disabled by default so opening the notebook does not start it.

For Colab, set `BASE_PATH_OVERRIDE` in the setup cell to the actual mounted
`CS682_Neural_Project` folder containing `01_data`. The default is an example;
check its location in the file browser after Drive mounting. Keep the override
at the project root, without appending `/01_data`. See the README Colab setup.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_ROOT_OVERRIDE = None
INSTALL_DEPENDENCIES = IN_COLAB
BASE_PATH_OVERRIDE = "/content/drive/MyDrive/CS682_Neural_Project" if IN_COLAB else None

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                         if (p / "configs/experiment.yaml").is_file()), None)
    if PROJECT_ROOT is None and IN_COLAB:
        PROJECT_ROOT = Path("/content/CS682-Synthetic-Data-Project")
        if not PROJECT_ROOT.exists():
            subprocess.run(["git", "clone", "--branch", "experiment/b2-baselines",
                            "https://github.com/khushboopandey6/CS682-Synthetic-Data-Project.git",
                            str(PROJECT_ROOT)], check=True)
    if PROJECT_ROOT is None:
        raise RuntimeError("Set PROJECT_ROOT_OVERRIDE to the cloned project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements.txt")], check=True)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

from src.project_config import load_config, resolve_base_path
CONFIG_PATH = PROJECT_ROOT / "configs/experiment.yaml"
config = load_config(CONFIG_PATH)
BASE_PATH = resolve_base_path(config, BASE_PATH_OVERRIDE)
print("Protocol:", config["protocol_id"], "| Status:", config["status"])
print("Artifact base:", BASE_PATH)

In [ ]:
from src.training import select_device
from src.classifier_pilot import run_classifier_pilot
from src.baseline_runs import run_baselines

print("Selected device:", select_device(config["training"]["device"]))
if not (BASE_PATH / "01_data/manifest.json").is_file():
    raise RuntimeError("Prepare or restore the official data release using notebook 01 first.")
run_classifier_pilot(CONFIG_PATH, BASE_PATH)

## Run the classifier pilot

Verify the configured artifact folder before starting. Matching completed pilot
fits are reused. Interrupted fits restart from their original seed. The official
test set is not read. Review validation curves before freezing, especially fits
still improving at epoch 30. Results are separate from the core matrix.


In [ ]:
RUN_CLASSIFIER_PILOT = False

if RUN_CLASSIFIER_PILOT:
    selection = run_classifier_pilot(CONFIG_PATH, BASE_PATH, execute=True, freeze=False)
else:
    print("Pilot preview only. Set RUN_CLASSIFIER_PILOT = True when storage and runtime are ready.")


In [ ]:
FREEZE_REVIEWED_PILOT = False

if FREEZE_REVIEWED_PILOT:
    selection = run_classifier_pilot(CONFIG_PATH, BASE_PATH, execute=False, freeze=True)
else:
    print("After reviewing all 12 fits, set FREEZE_REVIEWED_PILOT = True to freeze the shared rate.")


## Run Module A's main B1/B2/B3 matrix

The trainer refuses main runs until the pilot configuration is frozen. Start with
low-resource seed 42 for the first handoff, then expand to both settings and all
three seeds. Balanced B2 reuses B1 with a separate, traceable result entry.
Only validation predictions are exported here. Final test evaluation happens
after the experimental protocol and method decisions are frozen.

In [ ]:
RUN_MAIN_BASELINES = False
MAIN_SETTINGS = ["low_resource"]
MAIN_SEEDS = [42]

run_baselines(CONFIG_PATH, BASE_PATH, execute=RUN_MAIN_BASELINES,
              settings=MAIN_SETTINGS, seeds=MAIN_SEEDS)
# For the complete matrix, use both settings and seeds [42, 123, 456].